# MINE4201 - SR - Taller 1 - Experimentación y optimización para K vecinos más cercanos

**Grupo 3**

Lina María Ojeda Amaya Código: 202112324

Diego Felipe Tolosa Código: 201413235

Juan Manuel Rivera López Código: 201534131

Johana Alejandra Rátiva Mora Código: 202513844

# Importación de librerías y datos

In [1]:
import numpy as np
import os
import pandas as pd
import random
from matplotlib import pyplot as plt
import time

from scipy.sparse import csr_matrix

from sklearn.preprocessing import normalize
from tqdm import tqdm

#Para garantizar reproducibilidad en resultados
seed = 10
random.seed(seed)
np.random.seed(seed)

In [2]:
base_path = "C:/Users/jmriv/OneDrive - Universidad de los andes/Semestres uniandes/2026-1/Sistemas de recomendación/Talleres"
links_path = os.path.join(base_path, "link.csv")
movies_path = os.path.join(base_path, "movie.csv")
ratings_path = os.path.join(base_path, "rating.csv")

In [3]:
ratings = pd.read_csv(ratings_path)
ratings.head()

,userId,movieId,rating,timestamp
0,1,2,3.5,2005-04-02 23:53:47
1,1,29,3.5,2005-04-02 23:31:16
2,1,32,3.5,2005-04-02 23:33:39
3,1,47,3.5,2005-04-02 23:32:07
4,1,50,3.5,2005-04-02 23:29:40


In [4]:
movies = pd.read_csv(movies_path)
movies.head()

,movieId,title,genres
0,1,Toy Story (1995),Adventure|Animation|Children|Comedy|Fantasy
1,2,Jumanji (1995),Adventure|Children|Fantasy
2,3,Grumpier Old Men (1995),Comedy|Romance
3,4,Waiting to Exhale (1995),Comedy|Drama|Romance
4,5,Father of the Bride Part II (1995),Comedy


In [4]:
test = ratings.groupby("userId", group_keys=False).sample(frac=0.2, random_state=42)
train = ratings.drop(test.index)

In [5]:
user_cat = train.userId.astype("category")
item_cat = train.movieId.astype("category")

user_ids = user_cat.cat.codes
item_ids = item_cat.cat.codes

R_train = csr_matrix(
    (train.rating, (user_ids, item_ids))
)

In [6]:
user_map = dict(zip(user_cat.cat.categories, range(len(user_cat.cat.categories))))
item_map = dict(zip(item_cat.cat.categories, range(len(item_cat.cat.categories))))

# Evaluación de similitud usuario-usuario

In [7]:
N = R_train.shape[0]

counts = np.diff(R_train.indptr)
sums = np.add.reduceat(R_train.data, R_train.indptr[:-1])
means = sums / counts

In [8]:
def predict_rating(
        R,
        userId,
        movieId,
        neighbours,
        sims,
        K,
        sim_threshold,
        means,
        weighting=False,
        B=None,
        beta=50,):

    order = np.argsort(sims)[::-1]

    neighbours = neighbours[order][:K]
    sims = sims[order][:K]

    # --- get ratings for the movie ---
    ratings = R[neighbours, movieId].toarray().ravel()

    # keep neighbours that rated the movie
    mask = ratings > 0
    neighbours = neighbours[mask]
    sims = sims[mask]
    ratings = ratings[mask]

    # similarity threshold
    mask = sims >= sim_threshold
    neighbours = neighbours[mask]
    sims = sims[mask]
    ratings = ratings[mask]

    if len(ratings) == 0:
        return np.nan

    # --- mean centered ratings ---
    centered = ratings - means[neighbours]

    # --- McLaughlin significance weighting ---
    if weighting:

        intersections = (B[userId] @ B[neighbours].T).toarray().ravel()

        weights = np.minimum(intersections / beta, 1.0)

        sims *= weights

    numerator = np.sum(sims * centered)
    denominator = np.sum(np.abs(sims))

    if denominator == 0:
        return np.nan

    pred = means[userId] + numerator / denominator

    return pred

In [9]:
def evaluate_rmse(
        R,
        test_df,
        neighbors_list,
        sims_list,
        K,
        sim_threshold,
        means,
        weighting=False,
        B=None):

    preds = []
    truths = []

    for row in tqdm(test_df.itertuples(), total=len(test_df)):

        if row.userId not in user_map or row.movieId not in item_map:
            continue

        user_idx = user_map[row.userId]
        movie_idx = item_map[row.movieId]
        rating = row.rating

        pred = predict_rating(
            R,
            user_idx,
            movie_idx,
            neighbors_list[user_idx],
            sims_list[user_idx],
            K,
            sim_threshold,
            means,
            weighting,
            B
        )

        if not np.isnan(pred):
            preds.append(pred)
            truths.append(rating)

    preds = np.array(preds)
    truths = np.array(truths)

    rmse = np.sqrt(np.mean((preds - truths) ** 2))

    return rmse

In [ ]:
results = []

B = R_train.copy()
B.data = np.ones_like(B.data)

for metric in tqdm(["cosine", "pearson", "jaccard"], desc="Similarity metric"):
    with np.load(f"Train_neighbours/users_{metric}_top100_neighbors.npz") as data:
        top_k_indices = data["indices"]
        top_k_sims = data["sims"]
    print("min sim:", top_k_sims.min())
    print("max sim:", top_k_sims.max())

    for K in tqdm([10, 20, 50, 100], desc="K neighbors", leave=False):

        for sim_threshold in tqdm( [0,0.1,0.2,0.3], desc="sim_threshold", leave=False):
            for weight in [True, False]:
                rmse = evaluate_rmse(
                    R_train,
                    test,
                    top_k_indices,
                    top_k_sims,
                    K,
                    sim_threshold,
                    means,
                    weight,
                    B
                )

                results.append((metric, K, sim_threshold, weight, rmse))
                print((metric, K, sim_threshold, weight, rmse))

Similarity metric:   0%|          | 0/3 [00:00<?, ?it/s]

min sim: 0.05408
max sim: 0.9224
